# Introducción práctica a RAG con LLMs

Vale, en la sesión anterior vimos cómo consumir LLMs mediante APIs. En esta sesión vamos a dar un paso natural: ¿qué ocurre cuando queremos que el modelo responda usando información que no está necesariamente en sus pesos o que pertenece a nuestra organización?

Ahí entra **RAG** (*Retrieval-Augmented Generation*). La idea es sencilla: antes de pedirle al LLM que responda, buscamos en una base documental los fragmentos más relevantes y se los pasamos como contexto. Así el modelo no responde únicamente desde su memoria interna, sino apoyándose en documentos concretos.

En este notebook construiremos un RAG sobre un PDF local de preguntas frecuentes internas de una empresa.

## Objetivos de la sesión

Al terminar esta sesión deberías entender:

- Qué problema resuelve RAG y qué problemas no resuelve.
- Qué diferencias hay entre responder con un LLM “a pelo” y responder con contexto recuperado.
- Cómo cargar documentos, trocearlos en chunks, generar embeddings e indexarlos.
- Cómo recuperar los chunks más relevantes para una pregunta.
- Cómo construir un prompt que obligue al modelo a responder solo con el contexto.
- Cómo devolver fuentes para auditar de dónde sale la respuesta.
- Cómo evaluar de forma básica si un RAG está funcionando.
- Cómo empaquetar el flujo en LangGraph cuando ya tenemos clara la lógica.

Nótese que el objetivo no es “usar LangChain porque sí”, sino entender las piezas que aparecen en cualquier arquitectura RAG, uses LangChain, LlamaIndex, Haystack o código propio.

## 0. Preparación del entorno

Si ya tienes el entorno preparado, puedes saltarte la celda de instalación. Si estás ejecutando esto en un entorno limpio, descoméntala.

Usaremos Gemini como modelo generativo y también embeddings de Google para representar los documentos en forma vectorial.

In [ ]:
# %pip install -q -U google-genai langchain langchain-core langchain-community langchain-google-genai langchain-text-splitters langgraph python-dotenv pypdf

Cargamos credenciales y definimos los modelos. Para la parte generativa usaremos un modelo rápido y económico. Para embeddings usaremos el modelo de embeddings de Gemini disponible para texto.

Si tu versión de `langchain-google-genai` no reconoce `models/gemini-embedding-001`, puedes cambiarlo por `models/embedding-001`. La idea conceptual es la misma: necesitamos un modelo que convierta texto en vectores.

In [1]:
import getpass
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

GENERATION_MODEL = "gemini-2.5-flash-lite"
EMBEDDING_MODEL = "models/gemini-embedding-001"

PDF_PATH = Path("inputs/faqs.pdf")
print("Modelo generativo:", GENERATION_MODEL)
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("PDF:", PDF_PATH.resolve())

Modelo generativo: gemini-2.5-flash-lite
Modelo de embeddings: models/gemini-embedding-001
PDF: /Users/albertorag/Documents/02. Trabajo/03. Pontia/01. Máster IA Generativa/03. Repositorio/pontia_ia25/sesion_03/inputs/faqs.pdf


In [2]:
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

llm = init_chat_model(GENERATION_MODEL, model_provider="google_genai")
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

## 1. ¿Por qué necesitamos RAG?

Antes de construir nada, veamos el problema. Supongamos que un usuario pregunta algo sobre procedimientos internos de una empresa. Un LLM generalista puede dar una respuesta razonable, pero no tiene por qué conocer las políticas concretas de nuestra organización.

Vamos a hacer una pregunta sin proporcionarle ningún documento. Esto nos sirve como baseline.

In [3]:
pregunta = "¿Cómo puedo acceder al portal de empleados desde casa?"

respuesta_sin_rag = llm.invoke(
    "Responde a la siguiente pregunta de un empleado de una empresa. "
    "Si no conoces el procedimiento concreto, responde de la forma más útil posible.\n\n"
    f"Pregunta: {pregunta}"
)

print(respuesta_sin_rag.content)

Claro, entiendo perfectamente tu pregunta. Acceder al portal de empleados desde casa es algo muy común hoy en día y es importante que sepas cómo hacerlo.

Aunque no conozco el procedimiento exacto de tu empresa, te puedo dar una respuesta muy útil que te servirá para encontrar la información que necesitas y resolver tu duda.

Aquí tienes los pasos y la información que suelo recomendar en estos casos:

**1. Lo primero y más importante: Busca la información oficial de tu empresa.**

*   **Intranet o página web interna:** Muchas empresas tienen una intranet o una sección específica en su página web interna dedicada a los empleados. Busca un enlace o sección que diga algo como "Portal del Empleado", "Recursos para Empleados", "Acceso Remoto" o "Trabajo desde Casa".
*   **Manual del empleado o políticas internas:** Si tu empresa tiene un manual del empleado o documentos de políticas, revisa si hay información sobre el acceso a herramientas o portales.
*   **Comunicados internos:** Busca cor

La respuesta puede sonar bien, pero aquí tenemos un problema: no sabemos si está usando el procedimiento real de nuestra empresa o si está inventando una respuesta genérica.

RAG intenta resolver esto metiendo una fase previa:

1. El usuario hace una pregunta.
2. Buscamos en nuestros documentos los fragmentos relevantes.
3. Pasamos esos fragmentos al LLM junto con la pregunta.
4. El LLM responde usando ese contexto.
5. Idealmente, devolvemos también las fuentes utilizadas.

Esto no elimina todos los riesgos, pero cambia el problema: ahora podemos auditar qué contexto ha visto el modelo.

## 2. Carga documental

Para que la sesión sea reproducible, vamos a usar un PDF local: `inputs/faqs.pdf`. Es una pequeña base de conocimiento con preguntas frecuentes internas.

En un proyecto real podríamos cargar datos desde PDFs, páginas web, Notion, SharePoint, Google Drive, bases de datos, tickets de soporte o documentación interna. La pieza importante es que todo acabe convertido en objetos `Document`, que tienen dos partes:

- `page_content`: el texto del documento.
- `metadata`: información adicional, como fuente, página, ruta, fecha, categoría, etc.

In [4]:
from langchain_community.document_loaders import PyPDFLoader

if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

print("Número de páginas cargadas:", len(docs))
print("Metadata de la primera página:", docs[0].metadata)
print("Primeros caracteres:\n")
print(docs[0].page_content[:700])

Número de páginas cargadas: 2
Metadata de la primera página: {'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240319110349', 'source': 'inputs/faqs.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}
Primeros caracteres:

Base de Conocimiento para Q&A Interno de la Empresa
Sección de Preguntas Frecuentes (FAQs) Detallada
Pregunta: ¿Cómo puedo acceder al portal de empleados desde casa?
Respuesta: 
    Para acceder al portal de empleados desde casa, sigue estos pasos:
    1. Asegúrate de tener una conexión a internet segura.
    2. Utiliza la VPN de la empresa para una conexión segura.
    3. Accede a la página del portal de empleados en: http://portal.empleados.com.
    4. Introduce tu nombre de usuario y contraseña.
    Si tienes problemas para acceder, contacta al soporte técnico.
Pregunta: ¿Qué debo hacer si olvido mi contraseña del correo electrónico corporativo?
Respuesta: 
    Si olvidas tu contraseña de


Nótese que todavía no hemos hecho RAG. Solo hemos cargado el documento. El siguiente paso es decidir cómo lo troceamos.

¿Por qué trocear? Porque normalmente no queremos meter documentos enteros en el prompt. Queremos recuperar solo las partes relevantes. Además, los embeddings funcionan mejor cuando representan unidades de información suficientemente concretas.

## 3. Chunking: trocear el documento

El chunking es una de las decisiones más importantes en RAG. Si los chunks son demasiado grandes, recuperamos mucho ruido. Si son demasiado pequeños, podemos perder contexto.

No existe un tamaño universalmente correcto. Depende del documento, del idioma, del caso de uso, del modelo de embeddings y de cómo vayas a generar la respuesta. Aquí usaremos chunks pequeños porque el PDF es corto y está formado por preguntas y respuestas.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)

splits = text_splitter.split_documents(docs)

# Añadimos un identificador simple a cada chunk para poder citarlo luego.
for i, doc in enumerate(splits):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(splits))
print("\nEjemplo de chunk:\n")
print(splits[0].page_content)
print("\nMetadata:", splits[0].metadata)

Número de chunks: 7

Ejemplo de chunk:

Base de Conocimiento para Q&A Interno de la Empresa
Sección de Preguntas Frecuentes (FAQs) Detallada
Pregunta: ¿Cómo puedo acceder al portal de empleados desde casa?
Respuesta: 
    Para acceder al portal de empleados desde casa, sigue estos pasos:
    1. Asegúrate de tener una conexión a internet segura.
    2. Utiliza la VPN de la empresa para una conexión segura.
    3. Accede a la página del portal de empleados en: http://portal.empleados.com.

Metadata: {'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240319110349', 'source': 'inputs/faqs.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'start_index': 0, 'chunk_id': 0, 'source_name': 'faqs.pdf'}


In [6]:
for doc in splits:
    print(f"Chunk {doc.metadata['chunk_id']} | Página {doc.metadata.get('page')} | Caracteres: {len(doc.page_content)}")

Chunk 0 | Página 0 | Caracteres: 451
Chunk 1 | Página 0 | Caracteres: 412
Chunk 2 | Página 0 | Caracteres: 492
Chunk 3 | Página 0 | Caracteres: 140
Chunk 4 | Página 1 | Caracteres: 468
Chunk 5 | Página 1 | Caracteres: 434
Chunk 6 | Página 1 | Caracteres: 226


Vale, con esto ya tenemos una colección de fragmentos. Pero buscar texto directamente por coincidencia exacta sería muy limitado. Queremos búsqueda semántica: que una pregunta como “no recuerdo la clave del mail” pueda recuperar el fragmento sobre “olvidé mi contraseña del correo electrónico corporativo”.

Para eso necesitamos embeddings.

## 4. Embeddings y vector store

Un embedding es una representación numérica de un texto. La idea es que textos con significado parecido acaben cerca en el espacio vectorial.

El flujo es:

1. Convertimos cada chunk en un vector.
2. Guardamos esos vectores en una base vectorial.
3. Cuando llega una pregunta, también la convertimos en vector.
4. Buscamos los chunks con vectores más parecidos al vector de la pregunta.

Para clase usaremos `InMemoryVectorStore`, que vive en memoria. En producción usaríamos algo persistente como Chroma, FAISS, Qdrant, Pinecone, PGVector, Elasticsearch, etc.

In [7]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(splits)

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 7
Primeros IDs: ['75f8348e-cd04-48b8-b3e1-61031a2cc202', '8951a996-c48c-44b4-8d26-6a262e8e932b', '7eac5286-0e1b-4e24-9cbb-c6fdbe3ffd84']


## 5. Retrieval: recuperar contexto relevante

Probemos solo la fase de recuperación, sin generación. Esto es importante: en RAG conviene inspeccionar qué documentos se están recuperando antes de culpar al LLM de una mala respuesta.

Si el retriever trae mal contexto, el modelo va a responder mal o va a tener que decir que no sabe.

In [8]:
def mostrar_documentos_recuperados(documentos):
    for i, doc in enumerate(documentos, start=1):
        print(f"Resultado {i}")
        print("Fuente:", doc.metadata.get("source_name"))
        print("Página:", doc.metadata.get("page"))
        print("Chunk:", doc.metadata.get("chunk_id"))
        print(doc.page_content)
        print("-" * 80)

pregunta = "¿Qué tengo que hacer si he olvidado la contraseña del correo corporativo?"
retrieved_docs = vector_store.similarity_search(pregunta, k=3)

mostrar_documentos_recuperados(retrieved_docs)

Resultado 1
Fuente: faqs.pdf
Página: 0
Chunk: 1
4. Introduce tu nombre de usuario y contraseña.
    Si tienes problemas para acceder, contacta al soporte técnico.
Pregunta: ¿Qué debo hacer si olvido mi contraseña del correo electrónico corporativo?
Respuesta: 
    Si olvidas tu contraseña del correo electrónico corporativo, puedes restablecerla siguiendo estos
pasos:
    1. Ve a la página de inicio de sesión del correo y haz clic en "Olvidé mi contraseña".
--------------------------------------------------------------------------------
Resultado 2
Fuente: faqs.pdf
Página: 0
Chunk: 2
2. Sigue las instrucciones para restablecerla, que pueden incluir responder a preguntas de
seguridad.
    3. Recibirás un enlace de restablecimiento de contraseña en tu correo alternativo o por SMS.
    Para más ayuda, contacta al soporte técnico.
Pregunta: ¿Cómo puedo solicitar vacaciones o días libres?
Respuesta: 
    Para solicitar vacaciones o días libres, utiliza el sistema de gestión de RRHH de la emp

Aquí ya podemos hacer una primera evaluación manual. Antes de generar ninguna respuesta, pregúntate:

- ¿El primer chunk recuperado contiene la respuesta?
- ¿Los chunks posteriores aportan información útil o solo ruido?
- ¿El valor de `k` es razonable?
- ¿El chunk está cortado de forma que se entiende?

Esta revisión es simple, pero en proyectos reales evita muchos errores.

## 6. Generación con contexto

Ahora sí: vamos a pasar los chunks recuperados al modelo. La clave está en el prompt. Queremos que el modelo haga tres cosas:

1. Responder usando solo el contexto.
2. Decir que no tiene información si el contexto no contiene la respuesta.
3. Citar las fuentes utilizadas.

Esto es fundamental. Un RAG que no puede citar sus fuentes es mucho más difícil de auditar.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un asistente de Q&A interno de una empresa. "
            "Responde en español usando únicamente el contexto proporcionado. "
            "Si el contexto no contiene la respuesta, di claramente: 'No tengo información suficiente en la base documental'. "
            "No inventes datos. Cita las fuentes usadas al final.",
        ),
        (
            "user",
            "Pregunta:\n{question}\n\nContexto recuperado:\n{context}\n\nRespuesta:",
        ),
    ]
)

In [10]:
def format_docs_with_sources(documentos):
    bloques = []
    for i, doc in enumerate(documentos, start=1):
        source = doc.metadata.get("source_name", "fuente desconocida")
        page = doc.metadata.get("page", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        bloques.append(
            f"[Fuente {i}: {source}, página {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(bloques)


def answer_with_rag(question: str, k: int = 3, show_context: bool = False):
    retrieved_docs = vector_store.similarity_search(question, k=k)
    context = format_docs_with_sources(retrieved_docs)

    if show_context:
        print("CONTEXTO RECUPERADO")
        print(context)
        print("=" * 80)

    messages = rag_prompt.invoke({"question": question, "context": context})
    answer = llm.invoke(messages)

    return {
        "question": question,
        "answer": answer.content,
        "context": retrieved_docs,
        "prompt": messages,
    }

In [11]:
result = answer_with_rag(
    "¿Qué tengo que hacer si he olvidado la contraseña del correo corporativo?",
    k=3,
    show_context=True,
)

print(result["answer"])

CONTEXTO RECUPERADO
[Fuente 1: faqs.pdf, página 0, chunk 1]
4. Introduce tu nombre de usuario y contraseña.
    Si tienes problemas para acceder, contacta al soporte técnico.
Pregunta: ¿Qué debo hacer si olvido mi contraseña del correo electrónico corporativo?
Respuesta: 
    Si olvidas tu contraseña del correo electrónico corporativo, puedes restablecerla siguiendo estos
pasos:
    1. Ve a la página de inicio de sesión del correo y haz clic en "Olvidé mi contraseña".

[Fuente 2: faqs.pdf, página 0, chunk 2]
2. Sigue las instrucciones para restablecerla, que pueden incluir responder a preguntas de
seguridad.
    3. Recibirás un enlace de restablecimiento de contraseña en tu correo alternativo o por SMS.
    Para más ayuda, contacta al soporte técnico.
Pregunta: ¿Cómo puedo solicitar vacaciones o días libres?
Respuesta: 
    Para solicitar vacaciones o días libres, utiliza el sistema de gestión de RRHH de la empresa:
    1. Entra en el sistema y busca la sección "Solicitudes de tiempo l

Ahora prueba una pregunta que sí está en el PDF, pero formulada de otra manera. Esto nos permite comprobar si la búsqueda semántica está funcionando más allá de coincidencias literales.

In [12]:
result = answer_with_rag(
    "Quiero pedir unos días libres. ¿Dónde tengo que hacerlo?",
    k=3,
)

print(result["answer"])

Para solicitar días libres, debes entrar en el sistema y buscar la sección "Solicitudes de tiempo libre". [Fuente 1]


Y ahora una pregunta que no debería estar en la base documental. Esta prueba es igual de importante que las preguntas que sí tienen respuesta.

Un buen RAG no solo debe responder cuando sabe: también debe saber decir que no tiene información suficiente.

In [13]:
result = answer_with_rag(
    "¿Cuál es el número de teléfono directo del departamento de soporte técnico?",
    k=3,
    show_context=True,
)

print(result["answer"])

CONTEXTO RECUPERADO
[Fuente 1: faqs.pdf, página 1, chunk 6]
2. Contacta al departamento de seguridad directamente si es urgente.
    Proporciona una descripción detallada del problema y su ubicación exacta.
    La seguridad es responsabilidad de todos, por lo que tu reporte es crucial.

[Fuente 2: faqs.pdf, página 0, chunk 1]
4. Introduce tu nombre de usuario y contraseña.
    Si tienes problemas para acceder, contacta al soporte técnico.
Pregunta: ¿Qué debo hacer si olvido mi contraseña del correo electrónico corporativo?
Respuesta: 
    Si olvidas tu contraseña del correo electrónico corporativo, puedes restablecerla siguiendo estos
pasos:
    1. Ve a la página de inicio de sesión del correo y haz clic en "Olvidé mi contraseña".

[Fuente 3: faqs.pdf, página 0, chunk 0]
Base de Conocimiento para Q&A Interno de la Empresa
Sección de Preguntas Frecuentes (FAQs) Detallada
Pregunta: ¿Cómo puedo acceder al portal de empleados desde casa?
Respuesta: 
    Para acceder al portal de empleados 

## 7. Experimentos con `k`, chunk size y recuperación

El parámetro `k` indica cuántos chunks recuperamos. Si `k` es muy bajo, quizá nos falte contexto. Si es muy alto, metemos ruido y encarecemos la llamada al modelo.

Vamos a comparar qué se recupera con distintos valores de `k`.

In [14]:
pregunta_experimento = "¿Cómo reporto un problema de seguridad en la oficina?"

for k in [1, 2, 4]:
    print(f"\n\nK = {k}")
    print("=" * 80)
    docs_k = vector_store.similarity_search(pregunta_experimento, k=k)
    mostrar_documentos_recuperados(docs_k)



K = 1
Resultado 1
Fuente: faqs.pdf
Página: 1
Chunk: 6
2. Contacta al departamento de seguridad directamente si es urgente.
    Proporciona una descripción detallada del problema y su ubicación exacta.
    La seguridad es responsabilidad de todos, por lo que tu reporte es crucial.
--------------------------------------------------------------------------------


K = 2
Resultado 1
Fuente: faqs.pdf
Página: 1
Chunk: 6
2. Contacta al departamento de seguridad directamente si es urgente.
    Proporciona una descripción detallada del problema y su ubicación exacta.
    La seguridad es responsabilidad de todos, por lo que tu reporte es crucial.
--------------------------------------------------------------------------------
Resultado 2
Fuente: faqs.pdf
Página: 1
Chunk: 5
3. Establecimiento de objetivos para el próximo año.
    Prepárate revisando tus metas previas, recopilando ejemplos de tu trabajo, y pensando en tus
objetivos futuros.
Pregunta: ¿Cómo reportar un problema de seguridad en el

Nótese que este tipo de inspección es más útil de lo que parece. Muchas mejoras de RAG no empiezan cambiando el LLM, sino ajustando:

- cómo limpias los documentos,
- cómo partes los chunks,
- qué metadatos guardas,
- cuántos chunks recuperas,
- si necesitas filtros,
- si necesitas reranking,
- si necesitas combinar búsqueda semántica con búsqueda por palabras clave.

## 8. Evaluación básica de un RAG

Evaluar RAG puede hacerse de forma sofisticada, pero empecemos con algo manual y entendible.

Para cada pregunta, queremos observar:

- **Retrieval:** ¿los chunks recuperados contienen la respuesta?
- **Groundedness:** ¿la respuesta se apoya en el contexto o inventa?
- **Completitud:** ¿responde a lo que se preguntó?
- **Citas:** ¿indica de dónde sale la información?
- **No respuesta:** si no hay información, ¿lo reconoce?

Vamos a crear un mini set de preguntas de prueba.

In [15]:
test_questions = [
    "¿Cómo accedo al portal de empleados desde casa?",
    "¿Qué pasos sigo si olvidé la contraseña del correo?",
    "¿Cómo puedo solicitar vacaciones?",
    "¿En qué consiste la evaluación de desempeño?",
    "¿Cómo reporto un problema de seguridad?",
    "¿Cuál es el teléfono de soporte técnico?",
]

for question in test_questions:
    print("PREGUNTA:", question)
    result = answer_with_rag(question, k=3)
    print(result["answer"])
    print("-" * 100)

PREGUNTA: ¿Cómo accedo al portal de empleados desde casa?
Para acceder al portal de empleados desde casa, debes seguir estos pasos:
1. Asegúrate de tener una conexión a internet segura.
2. Utiliza la VPN de la empresa para una conexión segura.
3. Accede a la página del portal de empleados en: http://portal.empleados.com.
4. Introduce tu nombre de usuario y contraseña.
Si tienes problemas para acceder, contacta al soporte técnico.

[Fuente 1: faqs.pdf, página 0, chunk 0]
[Fuente 3: faqs.pdf, página 0, chunk 1]
----------------------------------------------------------------------------------------------------
PREGUNTA: ¿Qué pasos sigo si olvidé la contraseña del correo?
Si olvidas tu contraseña del correo electrónico corporativo, puedes restablecerla siguiendo estos pasos:
1. Ve a la página de inicio de sesión del correo y haz clic en "Olvidé mi contraseña".
2. Sigue las instrucciones para restablecerla, que pueden incluir responder a preguntas de seguridad.
3. Recibirás un enlace de re

Si esto fuera un proyecto real, guardaríamos las preguntas, los chunks recuperados, la respuesta y una evaluación. Podríamos medir métricas como precisión@k, recall@k, groundedness o faithfulness. Pero incluso esta evaluación manual ya ayuda a detectar si el problema está en la recuperación o en la generación.

## 9. Añadiendo filtros por metadatos

Los metadatos son una de las piezas más infravaloradas de RAG. No todo tiene que resolverse con embeddings. Si sabemos que un documento viene de una página concreta, una fecha, una categoría, un área o un idioma, podemos usar filtros.

Nuestro PDF es pequeño, pero vamos a simular categorías para entender el patrón.

In [16]:
for doc in splits:
    content = doc.page_content.lower()
    if "contraseña" in content or "correo" in content or "portal" in content:
        doc.metadata["category"] = "it"
    elif "vacaciones" in content or "desempeño" in content or "rrhh" in content:
        doc.metadata["category"] = "rrhh"
    elif "seguridad" in content:
        doc.metadata["category"] = "seguridad"
    else:
        doc.metadata["category"] = "general"

vector_store = InMemoryVectorStore(embeddings)
_ = vector_store.add_documents(splits)

for doc in splits:
    print(doc.metadata["chunk_id"], doc.metadata["category"], doc.page_content[:80].replace("\n", " "))

0 it Base de Conocimiento para Q&A Interno de la Empresa Sección de Preguntas Frecuen
1 it 4. Introduce tu nombre de usuario y contraseña.     Si tienes problemas para acc
2 it 2. Sigue las instrucciones para restablecerla, que pueden incluir responder a pr
3 general 1. Entra en el sistema y busca la sección "Solicitudes de tiempo libre".     2. 
4 rrhh Base de Conocimiento para Q&A Interno de la Empresa     3. Tu supervisor revisar
5 seguridad 3. Establecimiento de objetivos para el próximo año.     Prepárate revisando tus
6 seguridad 2. Contacta al departamento de seguridad directamente si es urgente.     Proporc


In [17]:
pregunta_it = "¿Qué hago si no puedo entrar al correo?"

docs_filtrados = vector_store.similarity_search(
    pregunta_it,
    k=3,
    filter=lambda doc: doc.metadata.get("category") == "it",
)

mostrar_documentos_recuperados(docs_filtrados)

Resultado 1
Fuente: faqs.pdf
Página: 0
Chunk: 1
4. Introduce tu nombre de usuario y contraseña.
    Si tienes problemas para acceder, contacta al soporte técnico.
Pregunta: ¿Qué debo hacer si olvido mi contraseña del correo electrónico corporativo?
Respuesta: 
    Si olvidas tu contraseña del correo electrónico corporativo, puedes restablecerla siguiendo estos
pasos:
    1. Ve a la página de inicio de sesión del correo y haz clic en "Olvidé mi contraseña".
--------------------------------------------------------------------------------
Resultado 2
Fuente: faqs.pdf
Página: 0
Chunk: 2
2. Sigue las instrucciones para restablecerla, que pueden incluir responder a preguntas de
seguridad.
    3. Recibirás un enlace de restablecimiento de contraseña en tu correo alternativo o por SMS.
    Para más ayuda, contacta al soporte técnico.
Pregunta: ¿Cómo puedo solicitar vacaciones o días libres?
Respuesta: 
    Para solicitar vacaciones o días libres, utiliza el sistema de gestión de RRHH de la emp

Los filtros no sustituyen a la búsqueda semántica, pero la acotan. En una empresa esto suele ser muy útil: documentos de RRHH, legal, producto, soporte, ventas, políticas internas, etc.

En sistemas más avanzados, el propio LLM puede analizar la pregunta y decidir qué filtros aplicar. Eso es lo que se suele llamar *query analysis* o *query rewriting*.

## 10. Query analysis: que el modelo ayude a buscar

Hasta ahora usamos la pregunta del usuario directamente como query de búsqueda. A veces funciona, pero otras veces el usuario pregunta de forma ambigua o conversacional.

Podemos pedirle al modelo que transforme la pregunta en una query más útil y que, además, elija una categoría de búsqueda.

In [18]:
from typing import Literal
from pydantic import BaseModel, Field

class SearchQuery(BaseModel):
    query: str = Field(description="Query optimizada para buscar en la base documental")
    category: Literal["it", "rrhh", "seguridad", "general"] = Field(
        description="Categoría más probable de la pregunta"
    )

query_analyzer = llm.with_structured_output(SearchQuery)

analyzed = query_analyzer.invoke(
    "No recuerdo mi clave del email de la empresa. ¿Qué hago?"
)

analyzed

SearchQuery(query='recuperar clave email empresa', category='it')

In [19]:
def retrieve_with_query_analysis(question: str, k: int = 3):
    analyzed_query = query_analyzer.invoke(question)
    retrieved_docs = vector_store.similarity_search(
        analyzed_query.query,
        k=k,
        filter=lambda doc: doc.metadata.get("category") == analyzed_query.category,
    )
    return analyzed_query, retrieved_docs

analyzed_query, retrieved_docs = retrieve_with_query_analysis(
    "No recuerdo mi clave del email de la empresa. ¿Qué hago?"
)

print("Query analizada:", analyzed_query)
mostrar_documentos_recuperados(retrieved_docs)

Query analizada: query='recuperar clave email empresa' category='it'
Resultado 1
Fuente: faqs.pdf
Página: 0
Chunk: 1
4. Introduce tu nombre de usuario y contraseña.
    Si tienes problemas para acceder, contacta al soporte técnico.
Pregunta: ¿Qué debo hacer si olvido mi contraseña del correo electrónico corporativo?
Respuesta: 
    Si olvidas tu contraseña del correo electrónico corporativo, puedes restablecerla siguiendo estos
pasos:
    1. Ve a la página de inicio de sesión del correo y haz clic en "Olvidé mi contraseña".
--------------------------------------------------------------------------------
Resultado 2
Fuente: faqs.pdf
Página: 0
Chunk: 2
2. Sigue las instrucciones para restablecerla, que pueden incluir responder a preguntas de
seguridad.
    3. Recibirás un enlace de restablecimiento de contraseña en tu correo alternativo o por SMS.
    Para más ayuda, contacta al soporte técnico.
Pregunta: ¿Cómo puedo solicitar vacaciones o días libres?
Respuesta: 
    Para solicitar vaca

Esto ya es una mejora sobre el RAG básico, pero conviene no abusar. Cada paso con LLM añade coste, latencia y posibilidad de error. La pregunta práctica siempre es: ¿la mejora en recuperación compensa el coste adicional?

## 11. Encapsulando el RAG en LangGraph

Hasta ahora hemos creado funciones sueltas. Eso está perfecto para aprender. Pero cuando el flujo crece, suele interesar separar los pasos de forma explícita.

LangGraph nos permite modelar el RAG como un pequeño grafo con dos nodos:

1. `retrieve`: recupera documentos.
2. `generate`: genera la respuesta con esos documentos.

No hace falta usar LangGraph para hacer RAG, pero sí ayuda a estructurar aplicaciones con más pasos, memoria, streaming, herramientas o lógica condicional.

In [20]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph

class RAGState(TypedDict):
    question: str
    context: List[Document]
    answer: str
    sources: List[dict]


def retrieve(state: RAGState):
    retrieved_docs = vector_store.similarity_search(state["question"], k=3)
    return {"context": retrieved_docs}


def generate(state: RAGState):
    context = format_docs_with_sources(state["context"])
    messages = rag_prompt.invoke({"question": state["question"], "context": context})
    response = llm.invoke(messages)
    sources = [
        {
            "source": doc.metadata.get("source_name"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
        }
        for doc in state["context"]
    ]
    return {"answer": response.content, "sources": sources}


graph_builder = StateGraph(RAGState)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
rag_graph = graph_builder.compile()

/Users/albertorag/Documents/02. Trabajo/03. Pontia/01. Máster IA Generativa/03. Repositorio/pontia_ia25/sesion_03/.venv/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [21]:
result = rag_graph.invoke({"question": "¿Cómo puedo solicitar vacaciones?"})

print("Respuesta:\n", result["answer"])
print("\nFuentes:")
for source in result["sources"]:
    print(source)

Respuesta:
 Para solicitar vacaciones o días libres, sigue estos pasos:
1. Entra en el sistema y busca la sección "Solicitudes de tiempo libre".
2. Selecciona "Crear nueva solicitud" y completa el formulario.
3. Tu supervisor revisará y responderá a tu solicitud. Mantén seguimiento de tu solicitud en el sistema para ver su estado.

[Fuente 1: faqs.pdf, página 0, chunk 3]
[Fuente 3: faqs.pdf, página 1, chunk 4]

Fuentes:
{'source': 'faqs.pdf', 'page': 0, 'chunk_id': 3}
{'source': 'faqs.pdf', 'page': 0, 'chunk_id': 2}
{'source': 'faqs.pdf', 'page': 1, 'chunk_id': 4}


In [22]:
for step in rag_graph.stream(
    {"question": "¿Cómo reporto un problema de seguridad?"},
    stream_mode="updates",
):
    print(step)
    print("-" * 80)

{'retrieve': {'context': [Document(id='d5f46c67-7a58-4431-b902-a3d4fde63bab', metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240319110349', 'source': 'inputs/faqs.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'start_index': 782, 'chunk_id': 6, 'source_name': 'faqs.pdf', 'category': 'seguridad'}, page_content='2. Contacta al departamento de seguridad directamente si es urgente.\n    Proporciona una descripción detallada del problema y su ubicación exacta.\n    La seguridad es responsabilidad de todos, por lo que tu reporte es crucial.'), Document(id='8c9846d5-ec09-4b12-bf9c-3f2abbe385dd', metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20240319110349', 'source': 'inputs/faqs.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'start_index': 416, 'chunk_id': 5, 'source_name': 'faqs.pdf', 'category': 'seguridad'}, page_content='3. Establecimiento de objetivos 

## 12. ¿Y el modo conversacional?

Un RAG conversacional añade una dificultad: la pregunta actual puede depender del historial.

Ejemplo:

> Usuario: ¿Cómo puedo solicitar vacaciones?  
> Asistente: Debes entrar en el sistema de RRHH...  
> Usuario: ¿Y quién lo aprueba?

La segunda pregunta no tiene sentido sin la primera. Hay dos formas habituales de resolverlo:

1. **Reformular la pregunta:** convertir “¿Y quién lo aprueba?” en “¿Quién aprueba una solicitud de vacaciones?” antes de buscar.
2. **Tool calling:** dejar que el modelo decida cuándo llamar al retriever y con qué query.

Esto es muy útil, pero también más avanzado. Lo importante para esta sesión es que veas que el RAG básico no guarda memoria por sí solo. Hay que diseñarla.

## 13. Riesgos y buenas prácticas en RAG

RAG no es magia. Mejora el acceso a conocimiento externo, pero introduce sus propios riesgos:

- **Mal retrieval:** si recuperas chunks incorrectos, la respuesta será mala.
- **Chunks mal cortados:** si separas pregunta y respuesta, tabla y explicación, o título y contenido, pierdes calidad.
- **Contexto excesivo:** meter demasiados chunks aumenta coste y ruido.
- **Datos obsoletos:** el sistema puede responder con documentos antiguos si no gestionas versiones.
- **Prompt injection documental:** un documento puede contener instrucciones maliciosas del tipo “ignora las instrucciones anteriores”.
- **Privacidad:** indexar documentos sensibles requiere controles de acceso; no basta con meterlos todos en una vector store.
- **Falta de evaluación:** sin tests, no sabes si el sistema mejora o empeora al cambiar chunking, embeddings, `k` o prompt.

Una regla pragmática: antes de tocar el LLM, mira qué chunks se están recuperando.

## 14. Recapitulación

Hemos construido un RAG de extremo a extremo:

1. Cargamos un PDF local.
2. Lo partimos en chunks.
3. Generamos embeddings.
4. Guardamos los chunks en una vector store.
5. Recuperamos contexto relevante para una pregunta.
6. Generamos una respuesta usando solo ese contexto.
7. Añadimos fuentes para poder auditar la respuesta.
8. Probamos preguntas con y sin respuesta en la base documental.
9. Añadimos filtros por metadatos y query analysis.
10. Encapsulamos el flujo básico en LangGraph.

Quédate con la idea central: RAG no consiste en “meter documentos al modelo”. RAG consiste en diseñar bien el circuito de recuperación para que el modelo vea justo el contexto que necesita, responda de forma fundamentada y permita auditar de dónde sale la información.

## 15. Siguientes pasos

Si quisieras llevar esto a un sistema más serio, los siguientes pasos naturales serían:

- Persistir la vector store en lugar de usar memoria.
- Añadir control de acceso por usuario o departamento.
- Guardar versiones de documentos.
- Añadir búsqueda híbrida: semántica + palabras clave.
- Añadir reranking.
- Evaluar con un conjunto fijo de preguntas.
- Registrar trazas con LangSmith u otra herramienta de observabilidad.
- Construir una interfaz conversacional con memoria.

Pero todo eso se apoya en lo mismo que hemos hecho aquí: cargar, partir, indexar, recuperar, generar y evaluar.